## Longest track task
### Loading parameters

In [1]:
## Autoreload is used for reloading imports (here when change are done in track_builder during development)
%load_ext autoreload
%autoreload 2

import os

import pandas as pd

import track_builder as tb

data = r"D:\Stockage\ASTD"
parquet_path = "../data/"

year = 2019
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# Periods for load_periods
periods = {
    2019: months,
    2020: months
    }

### Loading functions
Load raw csv files and save dataframe as parquet file - (it's faster to load parquet files)\
If parquet already exists, load the parquet file as a dataframe

In [2]:
def load_data(parquet_file, source, year, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        data = pd.read_parquet(parquet_file)
        print(f"Loaded data {parquet_file}, parameters ignored")
    else:
        data = tb.load_astd_monthly(base_path=source, year=year, **kwargs)
        data.to_parquet(parquet_file)
        print(f"Loaded data {parquet_file} from {source}")

    return data


def load_tracks(parquet_file, data, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        tracks = pd.read_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file}, parameters ignored")
    else:
        tracks = tb.build_ship_tracks(data, **kwargs)
        tracks.to_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file} from {data}")

    return tracks


def load_periods(parquet_file, source, periods, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        period = pd.read_parquet(parquet_file)
        print(f"Loaded period {parquet_file}, parameters ignored")
    else:
        period = tb.load_astd_periods(base_path=source, periods=periods, **kwargs)
        period.to_parquet(parquet_file)
        print(f"Loaded period {parquet_file} from {source}")

    return period

### Import all segments of selected year

Optional: \
Import segments first and last day to build tracks\
optional because first and last day are automatically recovered in algo, this could potentially make the process faster

In [3]:
all_data = load_data(parquet_file=f'all_segments{year}.parquet', source=data, year=year, months=months, remove_nan_rows="default", usecols="default", progress=True)
all_data.sample(5)

# Optional
# spe_track = load_data(parquet_file = f'spefirstlast{year}.parquet', source = data, year = year, months = months, remove_nan_rows="default", usecols="default", sampling=[0, -1], progress=True)
# spe_track.sample(5)

Loaded data ../data/all_segments2019.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
4070581,5672,2019-02-12 06:42:35+00:00,Cyprus,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,887.842712,159,19.007776,62.204800
25730269,4992,2019-08-20 02:28:48+00:00,Norway,FS Ice Class 1C,Offshore supply ships,5000 - 9999 GT,0.646706,539,12.664139,66.024277
15445962,3449,2019-06-03 03:48:21+00:00,Norway,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,7.038241,540,6.120750,62.470135
37595674,1843,2019-11-24 09:52:30+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,4.663179,361,14.378583,67.289574
39553869,3089,2019-12-10 01:17:19+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,1.879686,89,-22.542938,63.984772


### Build ship tracks
Build tracks between each months' segments with last day of previous month and first day of next month

In [4]:
tracks = load_tracks(parquet_file=f'tracks{year}.parquet', data=all_data)
tracks.sample(5)

Loaded tracks ../data/tracks2019.parquet, parameters ignored


,month,segment_id,track_id
7887,2019-09,5692,4134
11130,2019-10,11732,5843
609,2019-01,11943,610
10065,2019-12,3453,5232
6743,2019-06,4353,3634


### Build tracks position

'Ghost' point removal:\
remove_unrealistic_points

Horizontal date lines jumps:\
mask_dateline_jumps

In [6]:
# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region="norway", preprocess_positions=True)
display(build_tracks)

C:\Users\virtu\AppData\Local\Temp\ipykernel_28352\1944144291.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)


computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 1420 'ghost' or aberrant points removed.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:647: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[::point_stride])


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id
0,3250,2019-01-01 00:00:28+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,258.349396,1071,16.574747,71.363518,2019-01,38
1,3250,2019-01-01 03:37:43+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,1037.585083,1728,16.625444,71.310081,2019-01,38
2,3250,2019-01-01 07:14:37+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,3543.850830,1625,16.768854,71.284744,2019-01,38
3,3250,2019-01-01 09:57:09+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,121.411858,75,16.614000,71.428078,2019-01,38
4,3250,2019-01-01 15:35:42+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,2151.742920,588,16.635733,71.376556,2019-01,38
...,...,...,...,...,...,...,...,...,...,...,...,...
30626,6771,2019-12-31 20:40:44+00:00,Portugal (Mar),FS Ice Class 1A,Chemical tankers,5000 - 9999 GT,993.386353,161,20.014999,62.494667,2019-12,6927
30627,6771,2019-12-31 21:35:15+00:00,Portugal (Mar),FS Ice Class 1A,Chemical tankers,5000 - 9999 GT,1844.178955,290,20.105499,62.675167,2019-12,6927
30628,6771,2019-12-31 22:23:13+00:00,Portugal (Mar),FS Ice Class 1A,Chemical tankers,5000 - 9999 GT,2493.447998,391,20.181999,62.835999,2019-12,6927
30629,6771,2019-12-31 23:00:54+00:00,Portugal (Mar),FS Ice Class 1A,Chemical tankers,5000 - 9999 GT,2130.104004,338,20.245333,62.961834,2019-12,6927


### Compute longest track with build and filtered tracks

In [17]:
def get_longest_tracks(tracks : pd.DataFrame, n_tracks : int =5):
    # Verify column name exists
    if not {'track_id', 'dist_nextpoint'}.issubset(tracks.columns):
        print("No tracks found")
        return None

    # Get the total distance for each tracks
    total_dist_ship = (
        tracks.groupby('track_id', as_index=False)['dist_nextpoint']
        .sum()
        .rename(columns={'dist_nextpoint': 'total_distance'})
        .sort_values('total_distance', ascending=False)
    )

    df_longest_tracks = total_dist_ship.head(n_tracks)
    longest_track_ids = df_longest_tracks['track_id'].to_list()

    print("Longest track segments")
    display(tracks[tracks['track_id'] == longest_track_ids[0]])

    print('Longest tracks')
    display(df_longest_tracks)

    return longest_track_ids


longest_track_ids = get_longest_tracks(build_tracks)

print("Unique total tracks", tracks['track_id'].nunique())
print("Unique filtered tracks", build_tracks['track_id'].nunique())

# Visualize the longest track
fig = tb.plot_ship_tracks(
    build_tracks,
    track_ids=longest_track_ids,
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=False)
fig.show()

Longest track segments


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id
18421,3397,2019-05-01 00:03:02+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,202.312592,41,30.218559,70.714745,2019-05,3210
18422,3397,2019-05-01 00:45:59+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,1501.048096,363,30.391588,70.687340,2019-05,3210
18423,3397,2019-05-01 01:40:19+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,302.208710,363,30.442560,70.621544,2019-05,3210
18424,3397,2019-05-01 02:28:39+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,192.368118,362,30.454697,70.606964,2019-05,3210
18425,3397,2019-05-01 03:17:23+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,150.646286,370,30.451931,70.602974,2019-05,3210
...,...,...,...,...,...,...,...,...,...,...,...,...
24649,3029,2019-12-31 19:11:10+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,1.524736,400,29.729424,70.637428,2019-12,3210
24650,3029,2019-12-31 20:13:49+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,1.883307,380,29.729382,70.637436,2019-12,3210
24651,3029,2019-12-31 21:15:20+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,1.115834,360,29.729364,70.637428,2019-12,3210
24652,3029,2019-12-31 22:17:19+00:00,Norway,FS Ice Class 1C,Fishing vessels,< 1000 GT,2.540834,390,29.729340,70.637436,2019-12,3210


Longest tracks


,track_id,total_distance
42,3210,2155501.500
28,1922,1690589.250
39,2930,1503424.875
32,2134,1093494.125
27,1903,986811.625


Unique total tracks 6979
Unique filtered tracks 72


In [18]:
# ship id segments by month
all_data[(all_data['shipid'] == 536) & (all_data['month'] == "2013-02")]

,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month
1165133,536,2013-02-01 00:02:48+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,6.639144,366,-6.725475,62.117195,2013-02
1165324,536,2013-02-01 00:08:54+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,5.344646,363,-6.725419,62.117249,2013-02
1165640,536,2013-02-01 00:14:57+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1.640870,362,-6.725488,62.117210,2013-02
1165827,536,2013-02-01 00:20:59+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,5.022588,370,-6.725502,62.117199,2013-02
1166007,536,2013-02-01 00:27:09+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,3.738240,368,-6.725465,62.117241,2013-02
...,...,...,...,...,...,...,...,...,...,...,...
2353667,536,2013-02-28 00:17:57+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,28953.310547,5779,-18.302792,63.238178,2013-02
2356745,536,2013-02-28 01:54:16+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,104807.617188,41981,-18.858217,63.308201,2013-02
2377380,536,2013-02-28 13:33:57+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,77532.109375,22684,-20.822098,63.644241,2013-02
2388644,536,2013-02-28 19:52:01+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,43445.500000,11379,-22.372242,63.747524,2013-02
